# Blur + Edge Detection — Combined HLS IP
Single-pass 5x5 Gaussian-Sobel IP (`blur_edge_0`). One DMA transfer per frame.

In [ ]:
from pynq import Overlay, allocate
import numpy as np
import cv2
import os
import time

In [ ]:
BITSTREAM   = "blur_edge_combined.bit"
INPUT_VIDEO = "input.mp4"
OUTPUT_DIR  = "output_frames"

IMG_HEIGHT = 480
IMG_WIDTH  = 640

AP_CTRL    = 0x00
BE_HEIGHT  = 0x10
BE_WIDTH   = 0x18
BE_THRESH  = 0x20

# Threshold for 5x5 Gaussian-Sobel (max mag ~48960); tune as needed
THRESHOLD  = 3000

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
print("Loading overlay...")
ol        = Overlay(BITSTREAM)
dma       = ol.axi_dma_0
blur_edge = ol.blur_edge_0
print("Overlay loaded.")
print(f"  blur_edge_0 : {blur_edge}")
print(f"  axi_dma_0   : {dma}")

In [ ]:
in_buf  = allocate(shape=(IMG_HEIGHT * IMG_WIDTH,), dtype=np.uint8)
out_buf = allocate(shape=(IMG_HEIGHT * IMG_WIDTH,), dtype=np.uint8)
print(f"Input  buffer: 0x{in_buf.physical_address:08X}")
print(f"Output buffer: 0x{out_buf.physical_address:08X}")

In [ ]:
def run_pipeline(gray_frame: np.ndarray) -> np.ndarray:
    in_buf[:] = gray_frame.flatten()
    in_buf.flush()

    blur_edge.write(BE_HEIGHT, IMG_HEIGHT)
    blur_edge.write(BE_WIDTH,  IMG_WIDTH)
    blur_edge.write(BE_THRESH, THRESHOLD)
    blur_edge.write(AP_CTRL,   0x01)

    dma.recvchannel.transfer(out_buf)
    dma.sendchannel.transfer(in_buf)
    dma.sendchannel.wait()
    dma.recvchannel.wait()

    out_buf.invalidate()
    return np.array(out_buf).reshape(IMG_HEIGHT, IMG_WIDTH)

In [ ]:
cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {INPUT_VIDEO}")

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps          = cap.get(cv2.CAP_PROP_FPS)
print(f"Video: {total_frames} frames @ {fps:.1f} fps")

frame_idx, times = 0, []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame.shape[0] != IMG_HEIGHT or frame.shape[1] != IMG_WIDTH:
        frame = cv2.resize(frame, (IMG_WIDTH, IMG_HEIGHT))

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    t0       = time.perf_counter()
    edge_map = run_pipeline(gray)
    times.append(time.perf_counter() - t0)

    output = frame.copy()
    output[edge_map == 255] = [0, 0, 255]

    cv2.imwrite(os.path.join(OUTPUT_DIR, f"frame_{frame_idx:05d}.png"), output)
    frame_idx += 1
    if frame_idx % 50 == 0:
        print(f"  Processed {frame_idx}/{total_frames} frames")

cap.release()
avg_ms = (sum(times) / len(times)) * 1000 if times else 0
print(f"\nDone. {frame_idx} frames saved to '{OUTPUT_DIR}/'")
print(f"Avg pipeline time per frame: {avg_ms:.2f} ms")

In [ ]:
OUTPUT_VIDEO = "output_edges.avi"
fourcc  = cv2.VideoWriter_fourcc(*'XVID')
writer  = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (IMG_WIDTH, IMG_HEIGHT), isColor=True)

for fname in sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith(".png")):
    writer.write(cv2.imread(os.path.join(OUTPUT_DIR, fname)))

writer.release()
print(f"Saved: {OUTPUT_VIDEO}")

In [ ]:
in_buf.freebuffer()
out_buf.freebuffer()
print("Buffers freed.")